# Phase 0 — Setup & getting comfortable with the device

**Goal of this notebook:** get the environment working and build a *feel* for the
hardware we'll run on. No model yet. Just: which device do I have, how do I move a
tensor onto it, and how do I see how much memory it's using?

On this Mac we have **MPS** (Apple GPU) or CPU — no CUDA. The real training happens
later on an **A100**. The device-selection snippet below is the same one we'll reuse
everywhere, so it transparently picks CUDA on the A100 and MPS here.

In [ ]:
import torch
print("torch version :", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("mps  available:", torch.backends.mps.is_available())

## The device-selection helper (reused project-wide)

Prefer CUDA (the A100), then MPS (this Mac's GPU), then CPU. Everything we build
takes a `device` argument so the *same code* runs in both places.

In [ ]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("using device:", device)

## Feel-check: put a tensor on the device and see the memory move

The plan asks for a 5-line script that creates a random tensor on the GPU and prints
the memory used. Memory accounting differs by backend, so this helper handles all three.

In [ ]:
def device_mem_mb(dev: torch.device) -> float | None:
    """Currently-allocated tensor memory in MB, or None if the backend can't report it."""
    if dev.type == "cuda":
        return torch.cuda.memory_allocated(dev) / 1e6
    if dev.type == "mps":
        return torch.mps.current_allocated_memory() / 1e6
    return None  # CPU: no per-device allocator counter

before = device_mem_mb(device)
# A 4096x4096 float32 tensor = 4096*4096*4 bytes = 67.1 MB
x = torch.randn(4096, 4096, device=device)
after = device_mem_mb(device)

print(f"tensor: {tuple(x.shape)}  dtype={x.dtype}")
print(f"theoretical size: {x.numel() * x.element_size() / 1e6:.1f} MB")
if before is not None:
    print(f"allocator: {before:.1f} MB -> {after:.1f} MB  (delta {after - before:.1f} MB)")
else:
    print("(CPU backend has no per-device allocator counter — that's expected)")

**What to notice:** the allocator delta should match the theoretical size (~67 MB).
This is the whole game later: weights + gradients + Adam moments + activations all
live in this same pool. On the A100 you'll watch this number climb as the model and
batch grow.

In [ ]:
# Feel-check: how fast is a big matmul on this device? (rough throughput sense)
import time
a = torch.randn(2048, 2048, device=device)
b = torch.randn(2048, 2048, device=device)
# warmup (kernels compile / caches warm on first call)
for _ in range(3):
    _ = a @ b
if device.type == "cuda":
    torch.cuda.synchronize()
elif device.type == "mps":
    torch.mps.synchronize()

t0 = time.perf_counter()
N = 20
for _ in range(N):
    c = a @ b
if device.type == "cuda":
    torch.cuda.synchronize()
elif device.type == "mps":
    torch.mps.synchronize()
dt = (time.perf_counter() - t0) / N

# one 2048^3 matmul ~ 2 * 2048^3 FLOPs
gflops = 2 * 2048**3 / dt / 1e9
print(f"avg {dt*1e3:.2f} ms/matmul  (~{gflops:.0f} GFLOP/s)")
del a, b, c

## Takeaways

- We know our device and how to read its memory.
- The `get_device()` helper is the one we'll reuse everywhere.
- **Next:** `01_data_pipeline.ipynb` — download Multi30k, train a BPE tokenizer, build
  the dataset + collate function with padding and masks, and *look at* every step.